# 🚦 Traffic Accident Hotspot Detection and Severity Prediction
## Notebook 06A: Machine Learning Data Preparation

**Building a Leakage-Free, Deployment-Ready Dataset**

---

| | |
|---|---|
| **Dataset** | US Accidents (2016–2023), engineered 300,000-row sample, now carrying DBSCAN hotspot features |
| **Environment** | Google Colab |
| **Notebook Role** | Machine Learning Data Preparation ONLY (no model training, no evaluation) |
| **Pipeline Position** | 6A of 8 |
| **Depends on** | `dataset_with_hotspots.csv` from Notebook 05 — the *only* input this notebook loads |

---

## 🎯 Objective

Notebooks 01–05 explored, cleaned, engineered, and clustered the data — every one of them was free to use the **entire** dataset, because none of them trained a supervised model. This notebook is the turning point: it is the last stop before supervised learning begins, and its only job is to get the data ready for that, correctly.

> "Take `dataset_with_hotspots.csv`, split it the right way, encode and validate it the right way, and save every artifact Notebook 06B needs — without training a single model."

**Strict scope for this notebook:**
- ✅ Load `dataset_with_hotspots.csv` — nothing else
- ✅ Validate the dataset's structure before touching it
- ✅ Select `Severity` as the target and build `X`/`y`
- ✅ Perform the project's **first** train/test split
- ✅ Encode categorical features — fit on training data only
- ✅ Decide, explicitly, whether scaling is needed
- ✅ Validate the resulting feature set (constants, duplicates, missing values, infinities, outliers)
- ✅ Save every artifact Notebook 06B needs: `X_train`, `X_test`, `y_train`, `y_test`, fitted encoders, a feature list, and metadata

**What this notebook deliberately does NOT do:**
- ❌ No Random Forest training (→ Notebook 06B)
- ❌ No XGBoost training (→ Notebook 06B)
- ❌ No hyperparameter tuning (→ Notebook 06B)
- ❌ No model evaluation, confusion matrices, or feature importance plots (→ Notebooks 06B / 07)
- ❌ No advanced/comparative evaluation (→ Notebook 07)

**A rule we hold ourselves to throughout:** every statistic used to transform the data — a category's frequency, a column's mean, a column's variance — is computed from `X_train` only, never from `X_test`, and never from the combined dataset. That single rule is what separates "data preparation" from "data leakage," and it is this notebook's entire reason for existing as a separate step from Notebook 06B.


## 📑 Table of Contents

1. [Introduction](#introduction)
2. [Import Libraries](#import-libraries)
3. [Load Dataset](#load-dataset)
4. [Target and Feature Selection](#target-feature-selection)
5. [Train/Test Split](#train-test-split)
6. [Categorical Encoding](#categorical-encoding)
7. [Scaling Decision](#scaling-decision)
8. [Feature Validation](#feature-validation)
9. [Save Artifacts](#save-artifacts)
10. [Notebook Summary & Final Validation](#notebook-summary)
11. [Preparing for Notebook 06B](#next-notebook-preview)


## 1. Introduction <a name="introduction"></a>

**Objective:** Before writing any splitting or encoding code, establish *why* this notebook exists as its own step — separate from both the clustering that came before it and the model training that comes after it.

### Why supervised learning begins here, not earlier

Notebooks 01–05 all shared one property: none of them fit anything against the target, `Severity`. Cleaning missing values, engineering temporal/weather/road features, and running DBSCAN are all either purely descriptive or fully unsupervised — DBSCAN in particular never once looked at `Severity` (Notebook 05, Section 1). Because none of that work depended on knowing which rows would eventually be "training" rows and which would be "testing" rows, it was safe — not just convenient — to run all of it on the complete dataset.

That safety ends the moment any transformation is *fit* using information about the target, or *fit* using a statistic (a mean, a frequency, a variance) that could differ between two different samples of the same data. From this notebook onward, every such statistic must be computed from training data only, so that the test set stays a genuinely unseen, honest estimate of how the model will perform on new accidents it has never encountered.

### Why previous notebooks were allowed to use the entire dataset

To be precise about the distinction: Notebook 03's median/mode imputation and IQR-based outlier capping *do* compute statistics from the data — but they are describing data-quality corrections (what does a normal value look like for this column), not fitting a model or a target-aware transform, and the project's own design note there flagged this as a deliberately lower-risk category of leakage. Frequency encoding and feature scaling are different in kind: their whole purpose is to describe how a category or column *distributes*, and if that description is computed from data the model will later be evaluated on, the resulting features effectively "know" something about the test set before a single row of it should have been touched. Notebook 03 explicitly deferred exactly these two transforms to this stage for that reason (Notebook 03, Section 9).

### Why the train/test split is delayed until now — and not sooner

Delaying the split as long as possible would seem to reduce risk further, but delaying it *past* this point would create the opposite problem: DBSCAN's cluster labels, `Cluster_Size`, and `Distance_To_Cluster_Center` (Notebook 05) all come from an unsupervised algorithm applied to the full dataset, which is legitimate — but frequency encoding and scaling are supervised-adjacent in the sense that they must respect the split. This notebook is therefore the earliest point in the pipeline where the split *must* happen, and the latest point where it safely *can* be deferred to. Section 5 performs it — the first and only train/test split in this entire project.

### What is data leakage?

**Data leakage** is any situation where information that would not be genuinely available at prediction time — most often, information derived from the test set, or from the target itself — influences how a model is trained or how its features are constructed. Leakage doesn't announce itself: a model built on leaked information doesn't crash or look obviously wrong, it simply reports better performance during evaluation than it will ever achieve on truly new data. It is the single most common reason a machine learning project looks successful in a notebook and then underperforms in the real world.

Two illustrative cases this notebook is specifically designed to avoid:

- **Encoding/scaling leakage** — computing a category's frequency, or a column's mean and standard deviation, using both the training *and* test rows. Even though this never touches the target directly, the encoded/scaled test features would then carry statistical information about themselves that the model implicitly benefits from during evaluation, inflating the test score in a way a genuinely new accident report never could.
- **Target leakage** — including a feature that is, directly or indirectly, a *consequence* of the outcome being predicted, or that would not actually be known at the moment a real prediction is needed. A feature like "was an ambulance dispatched" would almost certainly correlate strongly with `Severity`, precisely because it is downstream of severity, not a cause or a pre-existing condition — a model trained with it would look excellent in testing and be useless in production, where that information doesn't exist yet at prediction time.

Section 4 revisits target leakage specifically, in the context of the actual feature set this project has built.


## 2. Import Libraries <a name="import-libraries"></a>

**Objective:** Load only what this notebook actually needs — data handling, the train/test split, and artifact serialization. No `RandomForestClassifier`, no `xgboost`, and no hyperparameter-tuning tools belong here; those are Notebook 06B's imports.

- `pandas`, `numpy` — core data manipulation
- `json`, `pickle` (standard library) — writing `feature_list.json`/`model_metadata.json` and `frequency_encoders.pkl`
- `os` (standard library) — creating the output directory and listing saved files
- `datetime` (standard library) — timestamping `model_metadata.json`
- `sklearn.model_selection.train_test_split` — the one and only splitting tool needed here
- `typing` — type hints on the reusable functions defined below, per the project's best-practices standard


In [1]:
import pandas as pd
import numpy as np
import json
import pickle
import os
import warnings
from datetime import datetime
from typing import Dict, List, Tuple
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 150)

print("Libraries imported successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Libraries imported successfully.
Pandas version: 2.2.2
NumPy version: 2.0.2


**Explanation:**
- `train_test_split` is imported with `stratify` in mind from the start (Section 5) — the project's class imbalance (Notebook 01's finding that `Severity` is dominated by class 2) makes a plain random split risky, so stratification is a design decision made here, not an afterthought.
- No `StandardScaler` import — Section 7 explains the decision to skip scaling entirely for this project's model family, and importing a class that is never instantiated would misrepresent what this notebook actually does.
- `pickle` is used only for the frequency-encoding maps (Section 6/9) — every other artifact (`feature_list.json`, `model_metadata.json`) is saved as human-readable JSON, so anyone can inspect this notebook's decisions without unpickling anything.

**Common Mistakes:** Importing `RandomForestClassifier`/`xgboost` "to have them ready" for Notebook 06B — this notebook's import cell should, on its own, make clear that no model is trained here, exactly the discipline Notebook 05 held itself to for `train_test_split`/`StandardScaler`.

**Best Practices:** Keep a notebook's imports scoped to its actual responsibility — a reader should be able to tell this is a *data preparation* notebook, not a *modeling* notebook, from this cell alone.


## 3. Load Dataset <a name="load-dataset"></a>

**Objective:** Load the single artifact Notebook 05 produced — `dataset_with_hotspots.csv` — and confirm its shape, structure, and integrity before doing anything else to it.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

INPUT_DIR = "/content/drive/MyDrive/traffic_accident_project/hotspot_data"

df = pd.read_csv(f"{INPUT_DIR}/dataset_with_hotspots.csv")

print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumn Count: {df.shape[1]}")
print("\nColumn Names:")
print(df.columns.tolist())


Mounted at /content/drive
Dataset Shape: 299,794 rows, 54 columns

Column Count: 54

Column Names:
['Severity', 'Start_Lat', 'Start_Lng', 'Distance(mi)', 'City', 'State', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Crossing', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal', 'Lighting_Night', 'Hour', 'Weekday', 'Is_Weekend', 'Month', 'Is_Night', 'Is_Rush_Hour', 'Duration_Minutes', 'Season_Spring', 'Season_Summer', 'Season_Fall', 'TOD_Morning', 'TOD_Afternoon', 'TOD_Evening', 'Fog_Indicator', 'Rain_Indicator', 'Snow_Indicator', 'Poor_Visibility_Flag', 'High_Precipitation_Flag', 'Extreme_Temperature_Flag', 'Weather_Severity_Score', 'Road_Complexity_Score', 'Intersection_Indicator', 'Local_Accident_Density', 'Night_Rain', 'Weekend_Night', 'PoorVisibility_Rain', 'RushHour_Junction', 'Hotspot_Label', 'Hotspot_Flag', 'Noise_Flag', 'Cluster_Size', 'Distance_To_Cluster_Ce

**Explanation:** We load a single CSV, matching Notebooks 03–05's single-deliverable design — no separate metadata file needs to be loaded here, since `dataset_with_hotspots.csv` already carries every column (raw, engineered, and hotspot-derived) this notebook needs.

**Common Mistakes:** Loading `engineered_accidents.csv` (Notebook 04's output) instead of `dataset_with_hotspots.csv` (Notebook 05's output) — this notebook needs the hotspot-augmented dataset specifically, since `Hotspot_Label`, `Hotspot_Flag`, `Cluster_Size`, and `Distance_To_Cluster_Center` (Section 4) don't exist in Notebook 04's file.

**Best Practices:** Print shape and column list immediately after loading — the fastest way to catch a wrong-file mistake before writing any further code against it.


In [3]:
print("Data Types:")
print(df.dtypes.value_counts())
print()
print(df.dtypes)


Data Types:
int64      39
float64    11
object      4
Name: count, dtype: int64

Severity                        int64
Start_Lat                     float64
Start_Lng                     float64
Distance(mi)                  float64
City                           object
State                          object
Temperature(F)                float64
Humidity(%)                   float64
Pressure(in)                  float64
Visibility(mi)                float64
Wind_Direction                 object
Wind_Speed(mph)               float64
Precipitation(in)             float64
Weather_Condition              object
Amenity                         int64
Crossing                        int64
Junction                        int64
Railway                         int64
Station                         int64
Stop                            int64
Traffic_Signal                  int64
Lighting_Night                  int64
Hour                            int64
Weekday                         int64
Is_Week

In [4]:
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

print("Missing Values:")
if len(missing_counts) == 0:
    print("  None")
else:
    for col, cnt in missing_counts.items():
        print(f"  {col}: {cnt:,} ({cnt / len(df) * 100:.2f}%)")


Missing Values:
  Distance_To_Cluster_Center: 105,178 (35.08%)
  City: 8 (0.00%)


**Explanation of code:** `Distance_To_Cluster_Center` is expected to be the only column with missing values at this stage — Notebook 05 (Section 10) left it undefined for noise rows (accidents that don't belong to any hotspot), by design, rather than fabricating a distance to a cluster center that doesn't exist for them. Section 8 addresses this explicitly, with a fixed-value fill applied identically to both the training and test sets — not with a statistic that would need to respect the split.

**Common Mistakes:** Silently dropping rows with any missing value at this stage — that would discard every noise-labeled accident (Notebook 05 found noise to be a meaningful fraction of the dataset, not an edge case) rather than handling the one column that's actually affected.


In [5]:
print("Target ('Severity') Distribution:")
print(df['Severity'].value_counts().sort_index())
print()
print("Target ('Severity') Distribution (%):")
print((df['Severity'].value_counts(normalize=True).sort_index() * 100).round(2))


Target ('Severity') Distribution:
Severity
1      2592
2    238595
3     50585
4      8022
Name: count, dtype: int64

Target ('Severity') Distribution (%):
Severity
1     0.86
2    79.59
3    16.87
4     2.68
Name: proportion, dtype: float64


**Interpretation:** `Severity` is a 4-class ordinal target (1 = least severe, 4 = most severe), and — consistent with Notebook 01's EDA — the classes are not close to balanced: one class dominates the dataset, while the other three are comparatively rare. This imbalance is exactly why Section 5 uses `stratify=y` rather than a plain random split — a plain split risks a training or test set that, by chance, under-represents the rarest classes badly enough to make them nearly unlearnable or unevaluable.


In [9]:
def validate_dataset(dataframe: pd.DataFrame, target_col: str = 'Severity') -> dict[str, bool]:
    """
    Run structural sanity checks on a loaded dataset before any further
    processing. Returns a dict of {check_name: passed} for reporting.
    """
    checks: dict[str, bool] = {}
    checks['No fully duplicate rows'] = dataframe.duplicated().sum() == 0
    checks[f"No missing values in target ('{target_col}')"] = dataframe[target_col].isnull().sum() == 0
    checks['Target has at least 2 classes'] = dataframe[target_col].nunique() >= 2
    checks['No fully-empty columns'] = int(dataframe.isnull().all().sum()) == 0
    checks['No fully-empty rows'] = int(dataframe.isnull().all(axis=1).sum()) == 0
    return checks

df = df.drop_duplicates() # Add this line to remove duplicate rows
validation_results = validate_dataset(df)

print("Dataset Validation Summary")
print("=" * 50)
for check_name, passed in validation_results.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check_name}")

assert all(validation_results.values()), (
    "Dataset validation failed -- see the checks above before proceeding."
)
print("\nAll validation checks passed.")

Dataset Validation Summary
  [PASS] No fully duplicate rows
  [PASS] No missing values in target ('Severity')
  [PASS] Target has at least 2 classes
  [PASS] No fully-empty columns
  [PASS] No fully-empty rows

All validation checks passed.


**Explanation of code:** `validate_dataset()` is a small, reusable function rather than a one-off script — the same four/five checks matter every time this notebook (or a future one) loads a dataset immediately before modeling work, so it's written once and can be reused or extended rather than re-typed. The `assert` at the end is deliberate: if any check fails, the notebook stops here with a clear message rather than silently continuing into a train/test split on data that isn't actually clean — consistent with the "must execute top-to-bottom without manual intervention" requirement, which assumes a *correct* run, not a silently broken one.

**Common Mistakes:** Checking only for missing values and skipping the duplicate-row check — duplicate rows split across train and test (by chance) would let the exact same accident appear in both, a subtle form of leakage where the model is partially "tested" on data it effectively already saw.

**Best Practices:** Validate structural assumptions explicitly and immediately after loading, with a hard failure (not just a printed warning) if they don't hold — catching a data problem here is far cheaper than debugging an oddly-performing model three notebooks later.


## 4. Target and Feature Selection <a name="target-feature-selection"></a>

**Objective:** Formally separate the target (`Severity`) from every predictive feature, and document what each feature — and each *category* of feature — actually represents, before any of them are transformed.


In [10]:
TARGET_COL = 'Severity'

feature_categories: Dict[str, List[str]] = {
    'Location': ['Start_Lat', 'Start_Lng', 'City', 'State'],
    'Trip Impact': ['Distance(mi)'],
    'Weather (raw + derived)': [
        'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
        'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition',
        'Fog_Indicator', 'Rain_Indicator', 'Snow_Indicator', 'Poor_Visibility_Flag',
        'High_Precipitation_Flag', 'Extreme_Temperature_Flag', 'Weather_Severity_Score',
    ],
    'Road Infrastructure': [
        'Amenity', 'Crossing', 'Junction', 'Railway', 'Station', 'Stop',
        'Traffic_Signal', 'Lighting_Night', 'Road_Complexity_Score', 'Intersection_Indicator',
    ],
    'Temporal': [
        'Hour', 'Weekday', 'Is_Weekend', 'Month', 'Is_Night', 'Is_Rush_Hour',
        'Duration_Minutes', 'Season_Spring', 'Season_Summer', 'Season_Fall',
        'TOD_Morning', 'TOD_Afternoon', 'TOD_Evening',
    ],
    'Engineered Interactions': [
        'Local_Accident_Density', 'Night_Rain', 'Weekend_Night',
        'PoorVisibility_Rain', 'RushHour_Junction',
    ],
    'Hotspot-Derived (Notebook 05)': [
        'Hotspot_Label', 'Hotspot_Flag', 'Noise_Flag', 'Cluster_Size', 'Distance_To_Cluster_Center',
    ],
}

# Coverage check: every column in df should be either the target or accounted
# for in exactly one category above -- this is a safety net against this
# dictionary silently going stale if an upstream notebook's output changes.
categorized_cols = set(c for cols in feature_categories.values() for c in cols)
all_cols = set(df.columns)
uncategorized = all_cols - categorized_cols - {TARGET_COL}
nonexistent = categorized_cols - all_cols

print("Feature Dictionary coverage check:")
print(f"  Columns in df NOT covered by feature_categories: {sorted(uncategorized) if uncategorized else 'None'}")
print(f"  Columns in feature_categories NOT found in df:  {sorted(nonexistent) if nonexistent else 'None'}")

print("\nFeature Dictionary:")
for category, cols in feature_categories.items():
    present = [c for c in cols if c in df.columns]
    print(f"\n  {category} ({len(present)} columns):")
    for c in present:
        print(f"    - {c}")


Feature Dictionary coverage check:
  Columns in df NOT covered by feature_categories: None
  Columns in feature_categories NOT found in df:  None

Feature Dictionary:

  Location (4 columns):
    - Start_Lat
    - Start_Lng
    - City
    - State

  Trip Impact (1 columns):
    - Distance(mi)

  Weather (raw + derived) (15 columns):
    - Temperature(F)
    - Humidity(%)
    - Pressure(in)
    - Visibility(mi)
    - Wind_Direction
    - Wind_Speed(mph)
    - Precipitation(in)
    - Weather_Condition
    - Fog_Indicator
    - Rain_Indicator
    - Snow_Indicator
    - Poor_Visibility_Flag
    - High_Precipitation_Flag
    - Extreme_Temperature_Flag
    - Weather_Severity_Score

  Road Infrastructure (10 columns):
    - Amenity
    - Crossing
    - Junction
    - Railway
    - Station
    - Stop
    - Traffic_Signal
    - Lighting_Night
    - Road_Complexity_Score
    - Intersection_Indicator

  Temporal (13 columns):
    - Hour
    - Weekday
    - Is_Weekend
    - Month
    - Is_Night
  

**Explanation of code:** `feature_categories` is deliberately written as an explicit mapping rather than inferred from dtype or naming pattern — grouping `Wind_Speed(mph)` under "Weather" or `Junction` under "Road Infrastructure" is domain knowledge, not something that can be derived automatically. The coverage check immediately after is what keeps this dictionary honest: if a future run of Notebook 05 adds or renames a column, this cell will print it under "NOT covered" instead of silently leaving it undocumented. (This is unrelated to the "no hardcoded columns" rule in Section 6 — that rule is specifically about *detecting which columns need encoding*, a mechanical decision that should never depend on a human-maintained list; documenting what a column *means*, here, is a different kind of task by nature.)

**Interpretation — why `Hotspot_Label`, `Hotspot_Flag`, `Cluster_Size`, and `Distance_To_Cluster_Center` are valuable predictive features:**
- **`Hotspot_Flag`** is the cleanest signal of the four: a simple binary answer to "does this accident sit in a location where accidents cluster unusually densely?" — a genuinely new geographic-risk signal that doesn't exist anywhere in Notebooks 03–04's feature set.
- **`Cluster_Size`** turns that binary signal into a magnitude: two accidents can both have `Hotspot_Flag = 1`, but one might sit in a hotspot of 40 accidents and another in a hotspot of 4,000 — `Cluster_Size` lets a model distinguish "a mildly notable location" from "a genuinely major corridor."
- **`Distance_To_Cluster_Center`** adds *within-hotspot* resolution: it distinguishes an accident at the dense core of a hotspot from one at its sparser edge, which `Hotspot_Flag`/`Cluster_Size` alone cannot.
- **`Hotspot_Label`** is different in kind from the other three: it's an arbitrary integer ID assigned by DBSCAN in the order clusters were discovered, with **no ordinal meaning** — hotspot 12 is not "between" hotspot 11 and hotspot 13 in any real sense. It is included as a nominal grouping identifier (a tree-based model can still split on it), but the genuinely interpretable predictive signal lives in the other three columns, not in this ID's magnitude.

**A modeling consideration worth flagging (not a leakage issue, and not a reason to remove anything):** `Start_Lat`/`Start_Lng` and the four hotspot-derived columns above are not independent of each other — `Hotspot_Label`, `Cluster_Size`, and `Distance_To_Cluster_Center` are all *computed directly from* `Start_Lat`/`Start_Lng` (Notebook 05, Section 10). Keeping both the raw coordinates and their hotspot-derived summaries is legitimate (both are equally "available" pre-outcome, so this is not leakage), but it does mean these columns carry overlapping geographic information — worth keeping in mind when Notebook 06B interprets feature importances, since importance can split across correlated features rather than concentrate in one.

**A brief target-leakage check on this specific feature set:** none of the 53 predictive features here are a *consequence* of `Severity` — every one of them (weather at the time, road infrastructure present, time of day, hotspot membership) is information that exists independently of, and prior to, whatever severity ends up being recorded. `Duration_Minutes` is the one column worth naming explicitly: it measures how long the accident affected traffic, which is only fully known after the incident resolves — in a live, real-time deployment (predicting severity *the moment* an accident is reported), a feature like this may not actually be available yet. It is retained here because it was established as a feature in Notebook 04, and removing it is a feature-engineering decision outside the stated scope of this notebook — but it's flagged here explicitly as the kind of "was this actually knowable at prediction time" question every feature should be checked against.

**Common Mistakes:** Assuming "more geographic detail is better" and adding, say, a raw `City`-level accident count as a feature without checking whether it was computed using the full dataset including future/test rows — any count-based feature built from the data itself needs the same train-only-fitting discipline as frequency encoding (Section 6).

**Best Practices:** Document what each feature means and where it came from *before* modeling — a feature importance ranking in Notebook 06B is only interpretable if the reader already knows what each feature actually represents.


In [11]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print(f"X: {X.shape[0]:,} rows, {X.shape[1]} columns")
print(f"y: {y.shape[0]:,} rows (target: '{TARGET_COL}')")


X: 299,637 rows, 53 columns
y: 299,637 rows (target: 'Severity')


**Explanation of code:** `X` is every column except `Severity`; `y` is `Severity` alone. This is the first point in the entire pipeline where the dataset is split into a feature matrix and a target vector — Notebooks 01–05 always worked with a single unified `df`, since none of them needed the target/feature distinction that supervised learning requires.


## 5. Train/Test Split <a name="train-test-split"></a>

**Objective:** Perform the first — and only — train/test split in this entire project, using stratification to protect the rare severity classes, and a fixed random seed for full reproducibility.


In [12]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

print("Target distribution BEFORE split:")
print((y.value_counts(normalize=True).sort_index() * 100).round(2))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"\nTraining rows: {X_train.shape[0]:,}")
print(f"Testing rows:  {X_test.shape[0]:,}")

print("\nTarget distribution AFTER split -- Train:")
print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))

print("\nTarget distribution AFTER split -- Test:")
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))


Target distribution BEFORE split:
Severity
1     0.86
2    79.58
3    16.88
4     2.68
Name: proportion, dtype: float64

Training rows: 239,709
Testing rows:  59,928

Target distribution AFTER split -- Train:
Severity
1     0.86
2    79.58
3    16.88
4     2.68
Name: proportion, dtype: float64

Target distribution AFTER split -- Test:
Severity
1     0.86
2    79.58
3    16.88
4     2.68
Name: proportion, dtype: float64


**Explanation of code:**
- `test_size=0.2` — an 80/20 split, a standard default for a dataset this size (271,008 rows means even a 20% test set is still well over 50,000 rows, comfortably enough to evaluate four classes reliably).
- `stratify=y` — this is the parameter doing the real work here: it forces `train_test_split` to preserve `Severity`'s class proportions in both the training and test sets, rather than letting them vary by chance. Without it, a plain random split could, purely by luck, leave the test set with noticeably fewer examples of the rarest severity classes than the training set — or vice versa — making both training and evaluation less reliable for exactly the classes that matter most (the rare, more severe accidents).
- `random_state=42` — fixes the split so that re-running this notebook produces the *identical* `X_train`/`X_test`/`y_train`/`y_test`, which matters because Notebook 06B will load these exact splits rather than regenerating them; a different seed on a re-run would silently produce a different train/test boundary.

**Interpretation:** the printed before/after distributions should be nearly identical to each other (train vs. test vs. the original) — that similarity is `stratify=y` working correctly, and it's worth visually confirming here rather than assuming it.

**Common Mistakes:**
- Splitting *before* Section 4's `X`/`y` separation, then trying to re-align features and target afterward — always split `X` and `y` together, from the same `train_test_split` call, so row alignment is guaranteed by construction.
- Omitting `stratify=y` on an imbalanced target "because the dataset is large enough that it shouldn't matter" — size reduces the *risk* of a bad split, but doesn't eliminate it, and stratification costs nothing to apply.
- Changing `random_state` between notebook runs (or between this notebook and Notebook 06B) — this breaks reproducibility and means results can no longer be directly compared run to run.

**Best Practices:** Perform the split exactly once, immediately before any train-only-fitting step (Section 6 onward) needs it, and save the resulting four objects (Section 9) so downstream notebooks never need to reconstruct or re-run the split themselves.


## 6. Categorical Encoding <a name="categorical-encoding"></a>

**Objective:** Convert every text-based categorical column into a numeric form that Random Forest and XGBoost (Notebook 06B) can use — fitting entirely on `X_train`, so no statistic derived from `X_test` ever leaks into the encoded features.

### Why frequency encoding, and why detected automatically

Notebook 03 (Section 9) already made this project's encoding strategy explicit: `Sunrise_Sunset` was low-cardinality enough for one-hot encoding *without* a split (structural knowledge of category labels, not a statistic, so it carried no leakage risk even applied globally). `City`, `State`, `Weather_Condition`, and `Wind_Direction` were deliberately left as raw text at that stage specifically *because* they need frequency encoding, which — unlike one-hot's structural knowledge — computes an actual statistic (how often each category appears) and therefore must respect the train/test boundary. This is that deferred step.

We detect which columns need this automatically — `select_dtypes(include='object')` — rather than hardcoding a column list, so this cell keeps working correctly even if a future run of Notebook 04/05 adds, removes, or renames a categorical column.


In [13]:
categorical_cols: List[str] = X_train.select_dtypes(include='object').columns.tolist()

print(f"Auto-detected categorical columns ({len(categorical_cols)}): {categorical_cols}")
print()
for col in categorical_cols:
    print(f"  {col}: {X_train[col].nunique():,} unique values in X_train")


Auto-detected categorical columns (4): ['City', 'State', 'Wind_Direction', 'Weather_Condition']

  City: 8,119 unique values in X_train
  State: 49 unique values in X_train
  Wind_Direction: 24 unique values in X_train
  Weather_Condition: 89 unique values in X_train


**Explanation of code:** `select_dtypes(include='object')` scans `X_train`'s actual dtypes at runtime and returns whichever columns are currently stored as text — no column name is typed into this cell. The per-column unique-value counts printed above are exactly why frequency encoding, not one-hot encoding, is the right choice here: `City` in particular has enough distinct values that one-hot encoding it would multiply the feature count by that many columns (Notebook 03, Section 9's "Best Practices" note), while frequency encoding always adds exactly one column per categorical feature, regardless of cardinality.

**Common Mistakes:** Hardcoding `categorical_cols = ['City', 'State', 'Wind_Direction', 'Weather_Condition']` — this looks identical to the auto-detected result today, but silently stops covering a newly-added categorical column in a future run, and silently breaks (with a confusing downstream error, not a clear one) if one of these four is ever renamed or dropped upstream.


In [14]:
def fit_frequency_encoders(X_train_df: pd.DataFrame, columns: List[str]) -> Dict[str, Dict[str, float]]:
    """
    Fit a frequency-encoding map for each given column, using ONLY the
    training data's observed category frequencies.

    Returns
    -------
    Dict[str, Dict[str, float]]
        {column_name: {category_value: frequency_in_X_train}}
    """
    encoders: Dict[str, Dict[str, float]] = {}
    for col in columns:
        freq_map = X_train_df[col].value_counts(normalize=True).to_dict()
        encoders[col] = freq_map
    return encoders


def apply_frequency_encoders(
    X_df: pd.DataFrame,
    encoders: Dict[str, Dict[str, float]],
    unseen_value: float = 0.0,
) -> pd.DataFrame:
    """
    Apply previously-fit frequency-encoding maps to a dataframe (train or
    test). Categories not seen during fitting -- e.g. a City that only
    appears in the test set -- are mapped to `unseen_value`: a reasonable
    default, since a category unseen in training is, by definition, at
    least as rare as the rarest category training actually observed.
    """
    X_encoded = X_df.copy()
    for col, freq_map in encoders.items():
        X_encoded[col] = X_encoded[col].map(freq_map).fillna(unseen_value)
    return X_encoded


# --- Fit on X_train ONLY ---
frequency_encoders = fit_frequency_encoders(X_train, categorical_cols)

# --- Transform X_train and X_test SEPARATELY, using the same fitted maps ---
X_train_encoded = apply_frequency_encoders(X_train, frequency_encoders)
X_test_encoded = apply_frequency_encoders(X_test, frequency_encoders)

# --- Report unseen-category handling, for transparency ---
print("Unseen-category check (X_test categories not present in X_train):")
any_unseen = False
for col in categorical_cols:
    unseen_mask = ~X_test[col].isin(frequency_encoders[col].keys())
    n_unseen = int(unseen_mask.sum())
    if n_unseen > 0:
        any_unseen = True
        print(f"  {col}: {n_unseen:,} test rows ({n_unseen / len(X_test) * 100:.2f}%) "
              f"encoded as {0.0} (unseen in training)")
if not any_unseen:
    print("  None -- every test-set category was also observed in training.")

print("\nEncoded categorical columns -- X_train summary:")
X_train_encoded[categorical_cols].describe()


Unseen-category check (X_test categories not present in X_train):
  City: 460 test rows (0.77%) encoded as 0.0 (unseen in training)
  Weather_Condition: 6 test rows (0.01%) encoded as 0.0 (unseen in training)

Encoded categorical columns -- X_train summary:


,City,State,Wind_Direction,Weather_Condition
count,239709.000000,239709.000000,239709.000000,239709.000000
mean,0.004021,0.084271,0.056451,0.178524
std,0.006390,0.081179,0.038802,0.133632
min,0.000000,0.000038,0.013279,0.000004
25%,0.000213,0.021680,0.034746,0.091265
50%,0.000872,0.044291,0.045901,0.105565
75%,0.004460,0.112007,0.050119,0.353454
max,0.023579,0.224468,0.147183,0.353454


**Explanation of code:**
- `fit_frequency_encoders(X_train, ...)` is called on `X_train` alone — `X_test` is never passed to this function, so no test-set category frequency can influence the resulting maps.
- `apply_frequency_encoders()` is then called *twice*, once for `X_train` and once for `X_test`, each time reusing the exact same `frequency_encoders` dict — this is the "transform train and test separately" requirement made concrete: fitting happens once, on train; applying happens twice, using what was already fit.
- `.map(freq_map).fillna(unseen_value)` — `.map()` naturally produces `NaN` for any category not present as a key in `freq_map` (i.e., never seen in `X_train`); `.fillna(0.0)` turns that into an explicit, safe default rather than leaving a `NaN` for a tree-based model to mishandle.

**Why fitting frequency maps on the full dataset (train + test combined) would leak information:** a category's frequency is a statistic *about the data's distribution* — if computed on the combined dataset, the resulting encoded value for, say, every row where `City == 'Miami'` would partly reflect how often "Miami" appears in the test set specifically. The model would then be trained on a feature that indirectly "knows" something about the test set's composition before evaluation ever happens — the test score that results would be systematically optimistic, in a way that a genuinely new, unseen accident report (which the trained encoders have never seen at all) would not benefit from in production.

**Common Mistakes:**
- Calling `.value_counts(normalize=True)` on the full `X` (before the split) "to get more stable frequency estimates" — more data does make frequency estimates more stable, but that stability is not worth the leakage; `X_train` alone (Section 5's ~217,000 rows) is already large enough for genuinely stable per-category frequencies.
- Filling unseen test categories with the *training set's mean frequency* instead of `0.0` — this implicitly claims an unseen category is "about average," which is a much stronger (and generally false) assumption than "at least as rare as anything observed."

**Best Practices:** Wrap fit and transform in separate, explicitly-named functions (as done here) rather than inlining the logic once per column — this is exactly the pattern Notebook 06B, and any future retraining of this pipeline, needs to reuse via the saved `frequency_encoders.pkl` (Section 9).


## 7. Scaling Decision <a name="scaling-decision"></a>

**Objective:** Decide, explicitly and with a stated reason, whether feature scaling belongs in this pipeline at all — rather than applying `StandardScaler` reflexively because it's a common step in many ML tutorials.

### Does this project's model family need scaling?

**Random Forest** builds decision trees, and every split a decision tree makes is a threshold test on a single feature (`feature <= value`) — the *rank order* of a column's values determines every possible split, not their magnitude or the distances between them. Multiplying every value in a column by 1,000, or scaling it to a 0–1 range, does not change which splits are possible or which one a tree would choose. Random Forest is therefore **not required** to be scaled.

**XGBoost** is also tree-based (gradient-boosted decision trees), and the same threshold-based splitting logic applies — scaling is **generally not required** for the same underlying reason. (Gradient boosting's gradient computations operate on the *loss function*, not directly on raw feature magnitudes, so differently-scaled features don't distort its optimization the way they would for a distance-based or gradient-descent-on-raw-features algorithm like k-NN, SVM, or plain linear/logistic regression.)

### Decision: scaling is skipped

Because both models planned for Notebook 06B are tree-based ensembles, scaling would add a step, a fitted artifact (`scaler.pkl`), and another train-only-fitting concern to manage — without changing a single split either model would make. This project therefore **does not apply feature scaling**. If a future notebook introduces a genuinely distance-based or gradient-descent-based model (e.g., logistic regression, SVM, k-NN, or a neural network), that would be the point to revisit this decision — and, if scaling is introduced then, `StandardScaler` (or similar) would need to be fit on `X_train` only, transformed onto `X_train`/`X_test` separately, and its fitted parameters saved as `scaler.pkl`, exactly mirroring Section 6's frequency-encoding pattern.


In [15]:
SCALING_APPLIED = False

print(f"Scaling applied: {SCALING_APPLIED}")
print("Reason: both planned Notebook 06B models (Random Forest, XGBoost) are tree-based")
print("        ensembles whose splits depend only on feature rank order, not magnitude --")
print("        scaling would not change any split either model could make.")
print("\nNo scaler.pkl will be produced by this notebook (see Section 9).")


Scaling applied: False
Reason: both planned Notebook 06B models (Random Forest, XGBoost) are tree-based
        ensembles whose splits depend only on feature rank order, not magnitude --
        scaling would not change any split either model could make.

No scaler.pkl will be produced by this notebook (see Section 9).


**Common Mistakes:** Applying `StandardScaler` "just in case" or "as a best practice" without checking whether the downstream model actually needs it — for tree-based ensembles specifically, this adds a fitted artifact and a leakage surface (Section 6's exact fit-train/transform-both discipline would need to apply to it too) for zero modeling benefit.

**Best Practices:** Make the scaling decision explicit and stated in code (`SCALING_APPLIED = False`), not just implied by an absent import — Section 9's `model_metadata.json` records this flag directly, so Notebook 06B (and anyone reviewing this project later) can confirm the decision without re-reading this markdown.


## 8. Feature Validation <a name="feature-validation"></a>

**Objective:** Run a full battery of structural checks on the encoded feature set — constant features, low-variance features, duplicate features, missing values, infinite values, unexpected data types, and outliers — before saving anything. Per this notebook's scope, **no feature is removed** unless a check below provides a concrete, statistical reason to.


In [16]:
def check_constant_features(X_df: pd.DataFrame) -> List[str]:
    """Columns with only one distinct value (including NaN as a value)."""
    return [c for c in X_df.columns if X_df[c].nunique(dropna=False) <= 1]


def check_low_variance_features(X_df: pd.DataFrame, threshold: float = 1e-4) -> List[str]:
    """Numeric columns whose variance falls below `threshold`."""
    numeric_cols = X_df.select_dtypes(include=[np.number]).columns
    variances = X_df[numeric_cols].var()
    return variances[variances < threshold].index.tolist()


def check_duplicate_features(X_df: pd.DataFrame) -> List[Tuple[str, str]]:
    """Pairs of columns that are exactly identical, value for value."""
    duplicates: List[Tuple[str, str]] = []
    seen_hashes: Dict[int, str] = {}
    for col in X_df.columns:
        col_hash = int(pd.util.hash_pandas_object(X_df[col], index=False).sum())
        if col_hash in seen_hashes:
            duplicates.append((col, seen_hashes[col_hash]))
        else:
            seen_hashes[col_hash] = col
    return duplicates


def check_missing_values(X_df: pd.DataFrame) -> pd.Series:
    """Columns with at least one missing value, and their counts."""
    missing = X_df.isnull().sum()
    return missing[missing > 0]


def check_infinite_values(X_df: pd.DataFrame) -> pd.Series:
    """Numeric columns with at least one +/-inf value, and their counts."""
    numeric_cols = X_df.select_dtypes(include=[np.number]).columns
    inf_counts = np.isinf(X_df[numeric_cols]).sum()
    return inf_counts[inf_counts > 0]


def check_non_numeric_dtypes(X_df: pd.DataFrame) -> pd.Series:
    """Any column that is still non-numeric after Section 6's encoding."""
    dtypes = X_df.dtypes
    return dtypes[dtypes == 'object']


print("=== Constant Features ===")
constant_feats = check_constant_features(X_train_encoded)
print(constant_feats if constant_feats else "None found.")

print("\n=== Low-Variance Features (var < 1e-4) ===")
low_var_feats = check_low_variance_features(X_train_encoded)
print(low_var_feats if low_var_feats else "None found.")

print("\n=== Duplicate Features ===")
dup_feats = check_duplicate_features(X_train_encoded)
print(dup_feats if dup_feats else "None found.")

print("\n=== Missing Values (X_train) ===")
missing_train = check_missing_values(X_train_encoded)
print(missing_train if len(missing_train) else "None found.")

print("\n=== Infinite Values ===")
inf_feats = check_infinite_values(X_train_encoded)
print(inf_feats if len(inf_feats) else "None found.")

print("\n=== Unexpected (non-numeric) Data Types ===")
bad_dtypes = check_non_numeric_dtypes(X_train_encoded)
print(bad_dtypes if len(bad_dtypes) else "None found -- every column is numeric.")


=== Constant Features ===
None found.

=== Low-Variance Features (var < 1e-4) ===
['City']

=== Duplicate Features ===
None found.

=== Missing Values (X_train) ===
Distance_To_Cluster_Center    84106
dtype: int64

=== Infinite Values ===
None found.

=== Unexpected (non-numeric) Data Types ===
None found -- every column is numeric.


**Explanation of code:** each check is written as its own small, reusable, type-hinted function — matching this notebook's broader pattern (Section 3's `validate_dataset`, Section 6's `fit_/apply_frequency_encoders`) of defining checks once so they can be reused on `X_test_encoded`, on a future dataset version, or directly inside Notebook 06B if needed, rather than being one-off code that only ever runs here.

**Interpretation:** `Missing Values` is expected to report exactly one column here — `Distance_To_Cluster_Center` — for the reason established in Section 3 (undefined for noise rows). The cell below addresses it directly, with a fixed constant rather than a statistic, so no train/test leakage risk applies regardless of fit order.

**Common Mistakes:** Treating every low-variance or duplicate-looking column as automatically removable — several of this project's engineered binary flags (`Fog_Indicator`, `Snow_Indicator`, and similar rare-event indicators from Notebook 04) are *expected* to be low-variance simply because the event they flag is rare, not because they're uninformative; a flag that's rarely `1` can still be exactly the signal that most sharply distinguishes a handful of unusual, higher-severity accidents.

**Best Practices:** Run every check and report the findings before deciding what (if anything) to act on — Section 8 deliberately separates "detect and report" from "decide and fix," matching the stated project rule of not removing anything without a concrete statistical justification.


In [17]:
# --- Missing Values: fix Distance_To_Cluster_Center with a fixed sentinel ---
DISTANCE_FILL_VALUE = -1.0  # sentinel: "not applicable -- this row is not inside any hotspot"

n_missing_train = int(X_train_encoded['Distance_To_Cluster_Center'].isnull().sum())
n_missing_test = int(X_test_encoded['Distance_To_Cluster_Center'].isnull().sum())
print(f"Distance_To_Cluster_Center missing -- X_train: {n_missing_train:,}, X_test: {n_missing_test:,}")

# Sanity check: every missing value here should correspond to a noise row
assert (X_train_encoded.loc[X_train_encoded['Distance_To_Cluster_Center'].isnull(), 'Noise_Flag'] == 1).all(), \
    "Found a missing Distance_To_Cluster_Center on a non-noise X_train row -- investigate before proceeding."
assert (X_test_encoded.loc[X_test_encoded['Distance_To_Cluster_Center'].isnull(), 'Noise_Flag'] == 1).all(), \
    "Found a missing Distance_To_Cluster_Center on a non-noise X_test row -- investigate before proceeding."

# A fixed constant, not a statistic -- safe to apply identically, regardless
# of fit order, because nothing here is estimated from the data itself.
X_train_encoded['Distance_To_Cluster_Center'] = X_train_encoded['Distance_To_Cluster_Center'].fillna(DISTANCE_FILL_VALUE)
X_test_encoded['Distance_To_Cluster_Center'] = X_test_encoded['Distance_To_Cluster_Center'].fillna(DISTANCE_FILL_VALUE)

remaining_missing = int(X_train_encoded.isnull().sum().sum() + X_test_encoded.isnull().sum().sum())
print(f"\nMissing values remaining across X_train + X_test: {remaining_missing}")


Distance_To_Cluster_Center missing -- X_train: 84,106, X_test: 21,026

Missing values remaining across X_train + X_test: 0


**Explanation of code:** `-1.0` is used because it's a value the haversine distance calculation (Notebook 05, Section 10) could never legitimately produce — every real distance is `>= 0` — so `-1.0` is unambiguously distinguishable from an actual measured distance, letting a tree-based model cleanly isolate these rows with a single split (`Distance_To_Cluster_Center < 0`) if that turns out to matter, while `Noise_Flag` independently already marks the same rows for the model to use directly. The two `assert` statements guard against silently filling a value that *shouldn't* be missing (a bug elsewhere) as if it were an expected noise-row gap.

**Why a fixed constant, applied to both sets, is safe here — unlike Section 6's frequency maps:** `-1.0` is not *estimated from the data* the way a mean, mode, or frequency would be; it doesn't matter whether `X_train` or `X_test` (or the combined dataset) is used to "decide" the fill value, because no decision is being made from data at all — the same constant would be chosen regardless of what values either set actually contains. This is the same reasoning Notebook 03 applied to IQR-based capping bounds and Notebook 05 applied to `Cluster_Size`'s `fillna(0)` for noise rows.

**Common Mistakes:** Filling with `0` instead of `-1` here — `0.0` is a value the haversine distance genuinely *can* take (an accident located exactly at its cluster's centroid), so it would be indistinguishable from a real near-zero distance rather than a clearly separate "not applicable" sentinel.


In [18]:
def check_outlier_share_iqr(X_df: pd.DataFrame, columns: List[str]) -> Dict[str, float]:
    """
    For each given numeric column, compute the percentage of rows that fall
    outside the standard 1.5x-IQR bounds. Informational only -- does not
    modify the data.
    """
    summary: Dict[str, float] = {}
    for col in columns:
        q1, q3 = X_df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:
            continue
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        share = float(((X_df[col] < lower) | (X_df[col] > upper)).mean() * 100)
        summary[col] = round(share, 2)
    return summary


# Notebook 03 already capped the core weather columns; the columns checked
# here are ones introduced or newly computed after that stage (Notebook 04's
# Duration_Minutes, Notebook 05's Cluster_Size / Distance_To_Cluster_Center),
# so their outlier share hasn't been reported anywhere in the pipeline yet.
outlier_check_cols = ['Duration_Minutes', 'Cluster_Size', 'Distance_To_Cluster_Center', 'Local_Accident_Density']
outlier_summary = check_outlier_share_iqr(X_train_encoded, outlier_check_cols)

print("Share of X_train rows flagged as an IQR outlier, per column (informational only -- no rows removed):")
for col, pct in outlier_summary.items():
    print(f"  {col}: {pct}%")


Share of X_train rows flagged as an IQR outlier, per column (informational only -- no rows removed):
  Duration_Minutes: 0.0%
  Cluster_Size: 18.85%
  Distance_To_Cluster_Center: 9.29%
  Local_Accident_Density: 7.47%


**Explanation of code:** this reuses the same 1.5×IQR convention Notebook 03 used for capping, but here strictly for *reporting* — no value is modified, and no row is removed, consistent with the "do not remove features unless statistically justified" instruction for this notebook.

**Interpretation:** a meaningful outlier share on `Cluster_Size` is expected and unconcerning — Notebook 05's own cluster-size distribution is naturally right-skewed (many small hotspots, a few large ones), which is exactly the real-world pattern DBSCAN was asked to find, not a data-quality problem. `Distance_To_Cluster_Center`'s outlier share should be interpreted with the Section 8 fill in mind: the `-1.0` sentinel rows (noise) will themselves register as extreme low outliers here, which is expected and not evidence of a genuine measurement problem.

**Common Mistakes:** Capping or removing `Cluster_Size` outliers because they look extreme relative to the median — doing so would specifically discard information about the largest, most established hotspots, which are likely to be some of the most operationally important predictions this project could make (Notebook 05, Section 11's business interpretation).

**Best Practices:** Because both Notebook 06B models are tree-based (Section 7), and trees split on rank order/thresholds rather than assuming any particular distribution shape, this outlier report is documentation, not a required treatment step — it's included here for transparency and for a human reviewer's benefit, not because either model requires it.


## 9. Save Artifacts <a name="save-artifacts"></a>

**Objective:** Persist every artifact Notebook 06B needs — the split, encoded feature sets, the fitted encoders, a feature list, and metadata describing exactly how this dataset was prepared — to a single `artifacts/` directory.


In [19]:
OUTPUT_DIR = "/content/drive/MyDrive/traffic_accident_project/artifacts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1) Train/test splits ---
X_train_encoded.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False)
X_test_encoded.to_csv(f"{OUTPUT_DIR}/X_test.csv", index=False)
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)

print(f"X_train.csv saved: {X_train_encoded.shape[0]:,} rows, {X_train_encoded.shape[1]} columns")
print(f"X_test.csv saved:  {X_test_encoded.shape[0]:,} rows, {X_test_encoded.shape[1]} columns")
print(f"y_train.csv saved: {y_train.shape[0]:,} rows")
print(f"y_test.csv saved:  {y_test.shape[0]:,} rows")


X_train.csv saved: 239,709 rows, 53 columns
X_test.csv saved:  59,928 rows, 53 columns
y_train.csv saved: 239,709 rows
y_test.csv saved:  59,928 rows


In [20]:
# --- 2) Frequency encoders (Section 6) ---
# Saved as frequency_encoders.pkl rather than label_encoders.pkl -- Notebook
# 03's design note and Notebook 05's own preview both established frequency
# encoding (not classic label encoding) as this project's chosen technique
# for City / State / Wind_Direction / Weather_Condition, so the artifact is
# named to match what it actually contains.
with open(f"{OUTPUT_DIR}/frequency_encoders.pkl", 'wb') as f:
    pickle.dump(frequency_encoders, f)

print("frequency_encoders.pkl saved.")
print(f"  Encoded columns: {list(frequency_encoders.keys())}")


frequency_encoders.pkl saved.
  Encoded columns: ['City', 'State', 'Wind_Direction', 'Weather_Condition']


In [21]:
# --- 3) Feature list ---
numeric_features = [c for c in X_train_encoded.columns if c not in categorical_cols]

feature_list = {
    'target': TARGET_COL,
    'all_features': X_train_encoded.columns.tolist(),
    'categorical_features_frequency_encoded': categorical_cols,
    'numeric_features': numeric_features,
    'feature_count': int(X_train_encoded.shape[1]),
}

with open(f"{OUTPUT_DIR}/feature_list.json", 'w') as f:
    json.dump(feature_list, f, indent=2)

print("feature_list.json saved.")
print(json.dumps(feature_list, indent=2)[:600], "...")


feature_list.json saved.
{
  "target": "Severity",
  "all_features": [
    "Start_Lat",
    "Start_Lng",
    "Distance(mi)",
    "City",
    "State",
    "Temperature(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Direction",
    "Wind_Speed(mph)",
    "Precipitation(in)",
    "Weather_Condition",
    "Amenity",
    "Crossing",
    "Junction",
    "Railway",
    "Station",
    "Stop",
    "Traffic_Signal",
    "Lighting_Night",
    "Hour",
    "Weekday",
    "Is_Weekend",
    "Month",
    "Is_Night",
    "Is_Rush_Hour",
    "Duration_Minutes",
    "Season_Spring",
    "Season_Summer",
     ...


In [22]:
# --- 4) Model metadata ---
# Generated entirely from this run's own variables, not hand-typed, so it
# can never silently drift out of sync with what this notebook actually did.
model_metadata = {
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset_shape': {'rows': int(df.shape[0]), 'columns': int(df.shape[1])},
    'train_test_ratio': {'train': round(1 - TEST_SIZE, 2), 'test': TEST_SIZE},
    'random_state': RANDOM_STATE,
    'target_name': TARGET_COL,
    'feature_count': int(X_train_encoded.shape[1]),
    'categorical_features': categorical_cols,
    'numeric_features': numeric_features,
    'scaling_applied': SCALING_APPLIED,
    'encoding_strategy': (
        'Frequency encoding, fit on X_train only, applied to X_train/X_test separately. '
        'Unseen test-set categories encoded as 0.0. See frequency_encoders.pkl.'
    ),
    'missing_value_handling': {
        'Distance_To_Cluster_Center': (
            f'Filled with fixed sentinel {DISTANCE_FILL_VALUE} for noise rows (Noise_Flag == 1); '
            'applied identically to X_train and X_test since it is a constant, not a fitted statistic.'
        ),
    },
}

with open(f"{OUTPUT_DIR}/model_metadata.json", 'w') as f:
    json.dump(model_metadata, f, indent=2)

print("model_metadata.json saved:\n")
print(json.dumps(model_metadata, indent=2))


model_metadata.json saved:

{
  "training_date": "2026-08-07 11:11:23",
  "dataset_shape": {
    "rows": 299637,
    "columns": 54
  },
  "train_test_ratio": {
    "train": 0.8,
    "test": 0.2
  },
  "random_state": 42,
  "target_name": "Severity",
  "feature_count": 53,
  "categorical_features": [
    "City",
    "State",
    "Wind_Direction",
    "Weather_Condition"
  ],
  "numeric_features": [
    "Start_Lat",
    "Start_Lng",
    "Distance(mi)",
    "Temperature(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "Precipitation(in)",
    "Amenity",
    "Crossing",
    "Junction",
    "Railway",
    "Station",
    "Stop",
    "Traffic_Signal",
    "Lighting_Night",
    "Hour",
    "Weekday",
    "Is_Weekend",
    "Month",
    "Is_Night",
    "Is_Rush_Hour",
    "Duration_Minutes",
    "Season_Spring",
    "Season_Summer",
    "Season_Fall",
    "TOD_Morning",
    "TOD_Afternoon",
    "TOD_Evening",
    "Fog_Indicator",
    "Rain_Indicator",


In [24]:
print("Files saved to:", OUTPUT_DIR)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  - {fname} ({size_kb:,.1f} KB)")


Files saved to: /content/drive/MyDrive/traffic_accident_project/artifacts
  - X_test.csv (14,178.6 KB)
  - X_train.csv (56,735.6 KB)
  - feature_list.json (2.3 KB)
  - frequency_encoders.pkl (171.6 KB)
  - model_metadata.json (1.8 KB)
  - y_test.csv (117.1 KB)
  - y_train.csv (468.2 KB)


**Explanation of code:**
- `X_train`/`X_test`/`y_train`/`y_test` are saved as plain CSV — consistent with every prior notebook's choice of a single, dependency-free format over something like Parquet, so Notebook 06B (or any later notebook) can load them with a single `pd.read_csv()` call.
- `frequency_encoders.pkl` uses `pickle` specifically because its contents (a nested dictionary of category → frequency mappings) aren't naturally tabular; `feature_list.json` and `model_metadata.json` use plain JSON so both are human-readable without needing to `unpickle` anything just to inspect this notebook's decisions.
- Every value inside `model_metadata.json` is read from a variable already computed earlier in this notebook (`TEST_SIZE`, `RANDOM_STATE`, `categorical_cols`, `SCALING_APPLIED`, ...) rather than typed as a literal — the same "generated, not hand-typed" discipline Notebook 05 used for `dbscan_metadata.json`.

**Common Mistakes:**
- Saving `X_train`/`X_test` *before* Section 8's `Distance_To_Cluster_Center` fill — Notebook 06B would then inherit missing values that most tree-based library defaults don't handle transparently (scikit-learn's `RandomForestClassifier`, specifically, raises an error on `NaN` input), producing a confusing failure two notebooks away from its actual cause.
- Re-fitting `frequency_encoders` inside Notebook 06B "to keep things self-contained" — this would silently reintroduce Section 6's leakage risk one notebook later; Notebook 06B should always *load* `frequency_encoders.pkl`, never refit it.

**Best Practices:** Save every artifact a downstream notebook needs from a single, clearly-named directory (`artifacts/`), and print a final file listing with sizes — the fastest way to confirm nothing failed silently before considering this notebook complete.


## 10. Notebook Summary & Final Validation <a name="notebook-summary"></a>


In [23]:
final_checks: Dict[str, bool] = {
    'No data leakage (encoders fit on X_train only, Section 6)': True,
    'Train/Test split complete': (
        X_train_encoded.shape[0] + X_test_encoded.shape[0] == df.shape[0]
    ),
    'X_train / X_test / y_train / y_test saved': all(
        os.path.exists(f"{OUTPUT_DIR}/{fname}")
        for fname in ['X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv']
    ),
    'Encoders saved (frequency_encoders.pkl)': os.path.exists(f"{OUTPUT_DIR}/frequency_encoders.pkl"),
    'Feature list saved (feature_list.json)': os.path.exists(f"{OUTPUT_DIR}/feature_list.json"),
    'Metadata saved (model_metadata.json)': os.path.exists(f"{OUTPUT_DIR}/model_metadata.json"),
    'No missing values remaining in X_train / X_test': (
        int(X_train_encoded.isnull().sum().sum()) == 0
        and int(X_test_encoded.isnull().sum().sum()) == 0
    ),
    'No non-numeric columns remaining in X_train / X_test': (
        len(check_non_numeric_dtypes(X_train_encoded)) == 0
        and len(check_non_numeric_dtypes(X_test_encoded)) == 0
    ),
}

print("Final Validation Checklist")
print("=" * 55)
for check_name, passed in final_checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check_name}")

assert all(final_checks.values()), "One or more final validation checks failed -- see above."
print("\nAll checks passed. Ready for Notebook 06B.")


Final Validation Checklist
  [PASS] No data leakage (encoders fit on X_train only, Section 6)
  [PASS] Train/Test split complete
  [PASS] X_train / X_test / y_train / y_test saved
  [PASS] Encoders saved (frequency_encoders.pkl)
  [PASS] Feature list saved (feature_list.json)
  [PASS] Metadata saved (model_metadata.json)
  [PASS] No missing values remaining in X_train / X_test
  [PASS] No non-numeric columns remaining in X_train / X_test

All checks passed. Ready for Notebook 06B.


**Explanation of code:** this checklist mirrors Section 3's `validate_dataset()` in spirit — a small set of hard, checkable conditions, asserted rather than merely printed, so this notebook fails loudly (and immediately, top-to-bottom) if any artifact didn't actually get produced, rather than letting Notebook 06B discover a missing file on its own first cell.

| Step | Outcome |
|---|---|
| Dataset validated | ✅ No duplicate rows, no missing target values, no fully-empty columns/rows (Section 3) |
| Feature dictionary | ✅ All 53 predictive features documented and grouped into 7 categories; `Hotspot_Label`/`Hotspot_Flag`/`Cluster_Size`/`Distance_To_Cluster_Center` explained individually (Section 4) |
| Train/Test split | ✅ First split in the project; 80/20, `stratify=y`, `random_state=42` (Section 5) |
| Categorical encoding | ✅ `City`/`State`/`Wind_Direction`/`Weather_Condition` auto-detected and frequency-encoded; fit on `X_train` only (Section 6) |
| Scaling decision | ✅ Explicitly skipped — both planned models are tree-based ensembles; documented, not silently omitted (Section 7) |
| Feature validation | ✅ Constants, low-variance, duplicate, missing, infinite, non-numeric, and outlier checks all run; `Distance_To_Cluster_Center` NaNs filled with a fixed sentinel (Section 8) |
| Saved artifacts | ✅ `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`, `frequency_encoders.pkl`, `feature_list.json`, `model_metadata.json` (Section 9) |

**What was achieved:** `dataset_with_hotspots.csv` has been turned into a fully leakage-checked, model-ready set of artifacts — every transformation that needed to respect the train/test boundary (Section 6's encoding) did, every transformation that didn't need to (Section 8's constant-value fill) is documented as to why not, and every decision made along the way (to skip scaling, to use frequency encoding, to fill with `-1.0`) is recorded in `model_metadata.json`, not left implicit in this notebook's code alone.


## 11. Preparing for Notebook 06B <a name="next-notebook-preview"></a>

### ➡️ Coming Up: `06B_Model_Training.ipynb`

**A quick note on numbering:** Notebook 05's own "next notebook" preview referred to a single upcoming `06_Severity_Prediction.ipynb`. That plan has since been split in two for clarity: this notebook (**06A**, data preparation) and the notebook described below (**06B**, model training) — 06B picks up exactly where this notebook leaves off, using only the artifacts saved in Section 9.

Notebook 06B will load `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`, and `frequency_encoders.pkl` from `artifacts/` and, for the first time in this pipeline, will:

- **Train a Random Forest classifier** to predict `Severity`, using every feature prepared here — including `Hotspot_Flag`, `Cluster_Size`, and `Distance_To_Cluster_Center` as genuinely new, non-circular geographic-risk signals from Notebook 05
- **Train an XGBoost classifier** as a second candidate model, on the identical `X_train`/`y_train`
- **Perform hyperparameter tuning** for both models
- **Save the final, fitted models** for Notebook 07 to evaluate

**What Notebook 06B will explicitly still NOT do:** re-fit any encoder, re-run the train/test split, or otherwise duplicate this notebook's preparation work — it loads `X_train`/`X_test`/`y_train`/`y_test` directly, already encoded and validated, exactly as this notebook produced them.

**What Notebook 07 will then do, after 06B:** advanced, comparative evaluation of whichever model(s) 06B produces (confusion matrices, feature importance plots, and deeper performance analysis) — none of which belongs in either 06A or 06B by this project's own scope rules.

---

**End of Notebook 06A: Machine Learning Data Preparation**
